# M1 · Create schemas & state tables (pure DDL)

**Goal:** run `sql/00_schemas_and_state.sql` only — nothing else. No dimension load, no fact load, no mart.

This is the strictly-DDL half of what the old `run_batch.py --mode bootstrap` did.

**Exit criterion:** `warehouse.etl_watermark`, `warehouse.etl_run_log`, `warehouse.quarantine_rejected` exist and the watermark rows are seeded with NULL.


## 1. Setup

In [1]:
# === Setup ===
from __future__ import annotations
import sys, pathlib
# Make the src/ package importable even if pip install -e was skipped.
PROJECT_ROOT = pathlib.Path().resolve().parents[1] if pathlib.Path().resolve().name != 'accent-fleet-analytics' else pathlib.Path().resolve()
for candidate in (PROJECT_ROOT, PROJECT_ROOT.parent):
    src = candidate / 'src'
    if src.exists() and str(src) not in sys.path:
        sys.path.insert(0, str(src))
        break

from accent_fleet.config import settings
from accent_fleet.db import get_engine, transaction
from sqlalchemy import text

s = settings()
print(f"DB = {s.pg_user}@{s.pg_host}:{s.pg_port}/{s.pg_database}")
print(f"Schemas: staging={s.pg_schema_staging}, warehouse={s.pg_schema_warehouse}, marts={s.pg_schema_marts}")


DB = medali_dev@localhost:15432/accent_data
Schemas: staging=staging, warehouse=warehouse, marts=marts


## 2. Inputs — the SQL we are about to execute

In [2]:
from accent_fleet.db.sql_loader import load_sql, split_sql_statements
sql = load_sql("00_schemas_and_state.sql")
statements = split_sql_statements(sql)
print(f"{len(statements)} statements found in 00_schemas_and_state.sql")
for i, stmt in enumerate(statements, 1):
    first_line = next((l for l in stmt.splitlines() if l.strip() and not l.strip().startswith('--')), '')
    print(f"  [{i:>2}] {first_line[:80]}")


11 statements found in 00_schemas_and_state.sql
  [ 1] CREATE SCHEMA IF NOT EXISTS staging
  [ 2] CREATE SCHEMA IF NOT EXISTS warehouse
  [ 3] CREATE SCHEMA IF NOT EXISTS marts
  [ 4] CREATE TABLE IF NOT EXISTS warehouse.etl_watermark (
  [ 5] CREATE TABLE IF NOT EXISTS warehouse.etl_run_log (
  [ 6] CREATE INDEX IF NOT EXISTS idx_etl_run_log_started
  [ 7] CREATE INDEX IF NOT EXISTS idx_etl_run_log_status
  [ 8] CREATE TABLE IF NOT EXISTS warehouse.quarantine_rejected (
  [ 9] CREATE INDEX IF NOT EXISTS idx_quarantine_rule
  [10] CREATE INDEX IF NOT EXISTS idx_quarantine_device
  [11] INSERT INTO warehouse.etl_watermark (layer, table_name, last_event_time, notes)


## 3. Execute

In [3]:
import time
t0 = time.time()
with transaction() as conn:
    for stmt in statements:
        conn.execute(text(stmt))
print(f"DDL applied in {time.time()-t0:.2f}s")


DDL applied in 3.70s


## 4. Inspect

In [4]:
import pandas as pd
with get_engine().connect() as conn:
    schemas = pd.read_sql(text("""
        SELECT schema_name FROM information_schema.schemata
        WHERE schema_name IN ('staging','warehouse','marts') ORDER BY schema_name
    """), conn)
    tables = pd.read_sql(text("""
        SELECT table_schema, table_name FROM information_schema.tables
        WHERE table_schema='warehouse' AND table_name IN
          ('etl_watermark','etl_run_log','quarantine_rejected')
        ORDER BY table_name
    """), conn)
    wm = pd.read_sql(text("SELECT layer, table_name, last_event_time FROM warehouse.etl_watermark ORDER BY table_name"), conn)
display(schemas); display(tables); display(wm)
assert len(schemas) == 3, "schemas missing"
assert len(tables) == 3, "state tables missing"
assert len(wm) >= 6, "watermark not seeded"
print("OK — schemas + state tables ready.")


,schema_name
0,marts
1,staging
2,warehouse


,table_schema,table_name
0,warehouse,etl_run_log
1,warehouse,etl_watermark
2,warehouse,quarantine_rejected


,layer,table_name,last_event_time
0,warehouse,dim_date,NaT
1,warehouse,dim_device,NaT
2,warehouse,dim_driver,NaT
3,warehouse,dim_hour_band,NaT
4,warehouse,dim_tenant,NaT
5,warehouse,dim_vehicle,NaT
6,warehouse,fact_daily_activity,2026-04-09 20:10:39
7,warehouse,fact_harsh_event,NaT
8,warehouse,fact_overspeed,2026-04-10 13:32:13
9,warehouse,fact_speed_notification,2026-04-10 13:32:24


OK — schemas + state tables ready.
